# Create Bronze Database & Ingest CSVs as Delta Tables

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS bronze")
print("Bronze database created")

Bronze database created


## Defining Tables & Volume Path

In [0]:
# Volume path
LANDING_PATH = "/Volumes/workspace/default/landing"

# All 8 tables 
TABLES = [
    "categories",
    "customers",
    "employees",
    "orders",
    "ordersdetails",
    "products",
    "shippers",
    "suppliers"
]

# Primary key per table - used as the merge condition in Delta MERGE operations.
PRIMARY_KEYS = {
    "categories":    "CategoryID",
    "customers":     "CustomerID",
    "employees":     "EmployeeID",
    "orders":        "OrderID",
    "ordersdetails": "OrderDetailID",
    "products":      "ProductID",
    "shippers":      "ShipperID",
    "suppliers":     "SupplierID"
}

# Ingesting All CSVs into Bronze Delta Tables via Incremental Loading

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.utils import AnalysisException
from datetime import datetime

# Pipeline run summary, tracks results across all tables

run_summary = []
run_start   = datetime.now()

for table in TABLES:
    pk          = PRIMARY_KEYS[table]
    table_start = datetime.now()
 
    # Read incoming CSV from Volumes
    df_new         = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(f"{LANDING_PATH}/{table}.csv")
    )
    incoming_count = df_new.count()
 
    try:
        # Table exists - capture row count before merge for comparison
        delta_table  = DeltaTable.forName(spark, f"bronze.{table}")
        count_before = delta_table.toDF().count()
 
        delta_table.alias("existing").merge(
            df_new.alias("incoming"),
            f"existing.{pk} = incoming.{pk}"
        ).whenMatchedUpdateAll() \
         .whenNotMatchedInsertAll() \
         .execute()
 
        # Capture row count after merge to calculate net new rows
        count_after = spark.table(f"bronze.{table}").count()
        new_rows    = count_after - count_before
        duration    = (datetime.now() - table_start).seconds
 
        print(f"bronze.{table}: {new_rows} new rows inserted | {count_after} total rows | {duration}s")
 
        run_summary.append({
            "table":      f"bronze.{table}",
            "status":     "merged",
            "new_rows":   new_rows,
            "total_rows": count_after,
            "duration_s": duration
        })
 
    except AnalysisException:
        # Table does not exist yet - create fresh, all rows are new
        (
            df_new.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(f"bronze.{table}")
        )
        duration = (datetime.now() - table_start).seconds
 
        print(f"bronze.{table}: created fresh | {incoming_count} rows loaded | {duration}s")
 
        run_summary.append({
            "table":      f"bronze.{table}",
            "status":     "created",
            "new_rows":   incoming_count,
            "total_rows": incoming_count,
            "duration_s": duration
        })

bronze.categories: 0 new rows inserted | 8 total rows | 31s
bronze.customers: 0 new rows inserted | 91 total rows | 11s
bronze.employees: 0 new rows inserted | 10 total rows | 9s
bronze.orders: 0 new rows inserted | 153 total rows | 9s
bronze.ordersdetails: 0 new rows inserted | 199 total rows | 9s
bronze.products: 0 new rows inserted | 77 total rows | 8s
bronze.shippers: 0 new rows inserted | 3 total rows | 8s
bronze.suppliers: 0 new rows inserted | 29 total rows | 7s


## Verify Bronze Tables

In [0]:
total_duration = (datetime.now() - run_start).seconds
total_new_rows = sum(r["new_rows"] for r in run_summary)
 
print("BRONZE LAYER - PIPELINE RUN REPORT")
print(f"Run completed at : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total duration   : {total_duration}s")
print(f"Total new rows   : {total_new_rows}")
print("="*10)
print(f"{'Table':<30} {'Status':<10} {'New Rows':<12} {'Total Rows':<12} {'Duration'}")
print("-"*10)
for r in run_summary:
    print(f"{r['table']:<30} {r['status']:<10} {r['new_rows']:<12} {r['total_rows']:<12} {r['duration_s']}s")
print("="*10)

BRONZE LAYER - PIPELINE RUN REPORT
Run completed at : 2026-03-30 09:07:56
Total duration   : 96s
Total new rows   : 0
Table                          Status     New Rows     Total Rows   Duration
----------
bronze.categories              merged     0            8            31s
bronze.customers               merged     0            91           11s
bronze.employees               merged     0            10           9s
bronze.orders                  merged     0            153          9s
bronze.ordersdetails           merged     0            199          9s
bronze.products                merged     0            77           8s
bronze.shippers                merged     0            3            8s
bronze.suppliers               merged     0            29           7s
